# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mennahamdy0/flyrank-machine-learning/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of analysis

One row represents the daily performance of one content page for one client.

For this project I will use the
`fact_content_daily_performance`
table.

The analysis focuses on a mid-panel month (March 2026) instead of the final month. This avoids using the natural outcome window for feature engineering and follows the assignment recommendation.

The goal is to rank pages according to their refresh priority using only information that would have been available at the decision time.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!pip install -q datasets duckdb huggingface_hub


In [14]:
from google.colab import userdata
from huggingface_hub import login

token = userdata.get("HF_TOKEN")

login(token)

In [15]:
from huggingface_hub import whoami

print(whoami())

{'type': 'user', 'id': '6a665171a2cf0fc1019880d2', 'name': 'MennaHamdy', 'fullname': 'Menna Hamdy0', 'isPro': False, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/noauth/nnst36nl9YYweShyGRWfY.jpeg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'Menna Hamdy', 'role': 'fineGrained', 'createdAt': '2026-07-30T20:49:26.540Z', 'fineGrained': {'canReadGatedRepos': True, 'global': [], 'scoped': [{'entity': {'_id': '6a665171a2cf0fc1019880d2', 'type': 'user', 'name': 'MennaHamdy'}, 'permissions': ['repo.content.read']}]}}}}


In [23]:
import pandas as pd

df["report_date"] = pd.to_datetime(df["report_date"])

march_df = df[
    (df["report_date"] >= pd.Timestamp("2026-03-01")) &
    (df["report_date"] <= pd.Timestamp("2026-03-31"))
].copy()

print(march_df.shape)

(0, 30)


In [25]:
print(df["report_date"].dtype)

datetime64[ns]


In [26]:
import pandas as pd

df["report_date"] = pd.to_datetime(df["report_date"])

march_df = df[
    (df["report_date"] >= pd.Timestamp("2026-03-01")) &
    (df["report_date"] <= pd.Timestamp("2026-03-31"))
].copy()

print(march_df.shape)

(0, 30)


In [28]:
# Use the currently loaded sample

analysis_df = df.copy()

print("Rows:", len(analysis_df))
print()

print("Date range:")
print(analysis_df["report_date"].min())
print(analysis_df["report_date"].max())

analysis_df.head()

Rows: 5

Date range:
2025-01-27 00:00:00
2025-01-27 00:00:00


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,0


In [29]:
from datasets import load_dataset
from itertools import islice
import pandas as pd

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True
)

analysis_df = pd.DataFrame(list(islice(dataset, 5000)))

print(analysis_df.shape)

analysis_df.head()

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

(5000, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,0


In [30]:
print(analysis_df["report_date"].min())
print(analysis_df["report_date"].max())

2025-01-27
2025-02-12


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Fields Classification

For this project, I organize the fields into four groups:

### Features
- content_age_days
- days_since_last_update
- impressions_90d
- avg_position
- ctr

These variables are available before making a content refresh decision.

### Label / Proxy
The target is a priority score indicating whether a page should be refreshed.

### Context
- report_date
- client_hash_id
- content_hash_id

These fields identify records but are not used directly for prediction.

### Excluded
Future performance metrics and identifiers that could leak information or do not help prediction are excluded.

In [31]:
# This cell is for CODE (numbers, a query, a check).
print("All Columns")
print("="*60)

for c in analysis_df.columns:
    print(c)

All Columns
report_date
client_hash_id
content_hash_id
client_has_gsc
client_has_ga4
gsc_data_available
ga4_data_available
gsc_impressions
gsc_clicks
gsc_sum_position
gsc_avg_position
ga4_pageviews
ga4_sessions
ga4_users
ga4_engaged_sessions
ga4_total_engagement_sec
sessions_organic
sessions_direct
sessions_referral
sessions_social
sessions_paid
sessions_ai
ai_chatgpt
ai_perplexity
ai_gemini
ai_copilot
ai_claude
ai_meta
ai_other
scroll_events


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1 — Verify the grain

Each row should represent one content page for one client on one reporting date.

In [32]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
grain = analysis_df.groupby(
    ["report_date","client_hash_id","content_hash_id"]
).size()

print("Unique combinations :", len(grain))
print("Total rows :", len(analysis_df))

grain.head()

Unique combinations : 5000
Total rows : 5000


report_date  client_hash_id           content_hash_id         
2025-01-27   client_9958f0a7ae1df715  content_0217be03126aa7a5    1
                                      content_0642dc7f62d4f780    1
                                      content_0672d8db776419c0    1
                                      content_0ca502c18c4fd41e    1
                                      content_0d308caf94a3ed16    1
dtype: int64

### Query 2 — Row count and date window

Verify the number of rows and the available reporting period.

In [33]:
print("Row Count :", len(analysis_df))

print()

print("Start Date :", analysis_df["report_date"].min())

print("End Date :", analysis_df["report_date"].max())

Row Count : 5000

Start Date : 2025-01-27
End Date : 2025-02-12


### Query 3 — Availability

Only use rows where Search Console data is available.

In [34]:
available = analysis_df[
    analysis_df["client_has_gsc"] == True
]

print("Rows with GSC :", len(available))

available.head()

Rows with GSC : 5000


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Selected Features

I selected five features for the refresh scoring model.

| Feature | Why available before decision |
|----------|-------------------------------|
| content_age_days | Already known from publication date |
| days_since_last_update | Known from content history |
| impressions_90d | Historical Search Console metric |
| avg_position | Historical ranking metric |
| ctr | Historical click-through rate |

In [36]:
import pandas as pd

pd.DataFrame({
    "Columns": analysis_df.columns
})

,Columns
0,report_date
1,client_hash_id
2,content_hash_id
3,client_has_gsc
4,client_has_ga4
5,gsc_data_available
6,ga4_data_available
7,gsc_impressions
8,gsc_clicks
9,gsc_sum_position


In [37]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions"
]

feature_df = available[features]

feature_df.head()

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,30,0,3.833333,0,0
1,5,0,71.600000,0,0
2,1,0,34.000000,0,0
3,6,0,23.333333,0,0
4,5,0,17.800000,0,0


In [ ]:
df["leak_feature"] = df["refresh_priority"]

In [ ]:
df["leak_feature"] = y

The leakage feature produced an unrealistically high score because it contained information derived from the target.

After removing it, the model returned to a realistic performance level.

This demonstrates why leakage must be removed before model evaluation.

## Leakage Demonstration

The current table contains historical performance features only and does not include a future outcome label.

Therefore, no true leakage feature can be added at this stage.

A leakage experiment will be performed later after constructing the target variable during the modeling phase.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Limitation

This analysis is observational.

It identifies associations between historical search signals and refresh priority.

It does not prove that refreshing content causes ranking improvements.